In [2]:
text = "ev ev evler evler evde güzel"

tokens = list(text.encode("utf-8"))

print("Karakter sayisi:", len(text))
print("Bayt sayisi    :", len(tokens))
print(tokens)

Karakter sayisi: 28
Bayt sayisi    : 29
[101, 118, 32, 101, 118, 32, 101, 118, 108, 101, 114, 32, 101, 118, 108, 101, 114, 32, 101, 118, 100, 101, 32, 103, 195, 188, 122, 101, 108]


In [4]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)

top_pair = max(stats, key=stats.get)
print("En sik ikili:", top_pair, "->", stats[top_pair], "kez")
print("Harf olarak :", chr(top_pair[0]), chr(top_pair[1]))

En sik ikili: (101, 118) -> 5 kez
Harf olarak : e v


In [5]:
def merge(ids, pair, new_id):
    new_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            new_ids.append(new_id)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

new_tokens = merge(tokens, top_pair, 256)

print("Once :", len(tokens), "token")
print("Sonra:", len(new_tokens), "token")
print(new_tokens)

Once : 29 token
Sonra: 24 token
[256, 32, 256, 32, 256, 108, 101, 114, 32, 256, 108, 101, 114, 32, 256, 100, 101, 32, 103, 195, 188, 122, 101, 108]


In [6]:
vocab = {i: bytes([i]) for i in range(256)}
merges = {}
ids = list(tokens)
num_merges = 10

for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    if stats[pair] < 2:
        print("Tekrar eden ikili kalmadı, duruyoruz.")
        break
    new_id = 256 + i
    ids = merge(ids, pair, new_id)
    merges[pair] = new_id
    vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
    print(f"{i + 1}. birleştirme: {vocab[new_id]} ({stats[pair]} kez) -> {new_id}")

print("Token sayısı:", len(tokens), "->", len(ids))

1. birleştirme: b'ev' (5 kez) -> 256
2. birleştirme: b' ev' (4 kez) -> 257
3. birleştirme: b' evl' (2 kez) -> 258
4. birleştirme: b' evle' (2 kez) -> 259
5. birleştirme: b' evler' (2 kez) -> 260
Tekrar eden ikili kalmadı, duruyoruz.
Token sayısı: 29 -> 14


In [8]:
def decode(ids):
    text_bytes = b"".join(vocab[idx] for idx in ids)
    text_out = text_bytes.decode("utf-8", errors="replace")
    return text_out

print(ids)
print(decode(ids))
print("Orijinalle ayni mi?", decode(ids) == text)

[256, 257, 260, 260, 257, 100, 101, 32, 103, 195, 188, 122, 101, 108]
ev ev evler evler evde güzel
Orijinalle ayni mi? True


In [ ]:
def encode(text):
    ids = list(text.encode("utf-8"))
    while len(ids) >= 2:
        stats = get_stats(ids)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        ids = merge(ids, pair, merges[pair])
    return ids

yeni = "evdlere ev var"
enc = encode(yeni)

print(enc)
print([vocab[i] for i in enc])
print("Orijinalle aynı mı?", decode(enc) == yeni)


[256, 108, 101, 114, 100, 101, 257, 32, 118, 97, 114]
[b'ev', b'l', b'e', b'r', b'd', b'e', b' ev', b' ', b'v', b'a', b'r']
Orijinalle aynı mı? True
